In [ ]:
"""week2_assignment.ipynb

# Week 2 Assignment — End-to-End ML Pipeline
## Tesla Deliveries & Price Data (2015–2025)

**Pipeline Stages:** Data Loading → Preprocessing → EDA → Feature Engineering → Regression Modeling → Hyperparameter Tuning → Time Series Forecasting

---
"""

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

In [ ]:
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')

In [ ]:
print("All imports OK")

In [ ]:
"""---
## Part 1 — Data Loading & Initial Exploration
"""

In [ ]:
# load the dataset
df = pd.read_csv('Docs/archive/tesla_deliveries_dataset_2015_2025.csv')

In [ ]:
print(f"Dataset shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
print(f"\nFirst 5 rows:")
print(df.head())

In [ ]:
# basic info
print(f"\n--- Dataset Info ---")
print(f"Total records: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nBasic statistics:")
print(df.describe())

In [ ]:
"""---
## Part 2 — Data Preprocessing
"""

In [ ]:
# check for duplicates
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")
if dupes > 0:
    df = df.drop_duplicates()
    print(f"After removing duplicates: {df.shape}")

In [ ]:
# check missing values again
print(f"\nMissing values per column:\n{df.isnull().sum()}")

In [ ]:
# fill any numeric nulls with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Filled {col} nulls with median: {median_val}")

In [ ]:
# check for any remaining nulls
print(f"\nTotal nulls remaining: {df.isnull().sum().sum()}")

In [ ]:
# create a proper date column for time series later
df['Date'] = pd.to_datetime(df[['Year', 'Month']].assign(Day=1))
# sort by date
df = df.sort_values('Date').reset_index(drop=True)
print(f"\nDate range: {df['Date'].min()} to {df['Date'].max()}")
print(f"\nPreprocessed data shape: {df.shape}")
print(df.head())

In [ ]:
"""---
## Part 3 — Exploratory Data Analysis (EDA)

### 3.1 Univariate Analysis
"""

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

In [ ]:
# distribution of key numeric features
axes[0, 0].hist(df['Estimated_Deliveries'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of Estimated Deliveries')
axes[0, 0].set_xlabel('Deliveries')

In [ ]:
axes[0, 1].hist(df['Avg_Price_USD'], bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution of Avg Price (USD)')
axes[0, 1].set_xlabel('Price')

In [ ]:
axes[0, 2].hist(df['Range_km'], bins=30, color='mediumseagreen', edgecolor='black', alpha=0.7)
axes[0, 2].set_title('Distribution of Range (km)')
axes[0, 2].set_xlabel('Range')

In [ ]:
axes[1, 0].hist(df['CO2_Saved_tons'], bins=30, color='mediumpurple', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('CO2 Saved (tons)')
axes[1, 0].set_xlabel('CO2 Saved')

In [ ]:
axes[1, 1].hist(df['Production_Units'], bins=30, color='goldenrod', edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Production Units')
axes[1, 1].set_xlabel('Units')

In [ ]:
axes[1, 2].hist(df['Charging_Stations'], bins=30, color='indianred', edgecolor='black', alpha=0.7)
axes[1, 2].set_title('Charging Stations')
axes[1, 2].set_xlabel('Stations')

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:
"""### 3.2 Categorical Analysis"""

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

In [ ]:
# deliveries by region
region_del = df.groupby('Region')['Estimated_Deliveries'].mean().sort_values()
region_del.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Avg Deliveries by Region')

In [ ]:
# deliveries by model
model_del = df.groupby('Model')['Estimated_Deliveries'].mean().sort_values()
model_del.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Avg Deliveries by Model')

In [ ]:
# avg price by model
model_price = df.groupby('Model')['Avg_Price_USD'].mean().sort_values()
model_price.plot(kind='barh', ax=axes[2], color='mediumseagreen')
axes[2].set_title('Avg Price by Model')

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:
"""### 3.3 Time Series Trends"""

In [ ]:
# monthly deliveries over time
monthly_data = df.groupby('Date').agg({
    'Estimated_Deliveries': 'sum',
    'Avg_Price_USD': 'mean',
    'Production_Units': 'sum'
}).reset_index()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

In [ ]:
axes[0].plot(monthly_data['Date'], monthly_data['Estimated_Deliveries'], color='steelblue', linewidth=1.5)
axes[0].set_title('Total Monthly Deliveries Over Time')
axes[0].set_ylabel('Deliveries')

In [ ]:
axes[1].plot(monthly_data['Date'], monthly_data['Avg_Price_USD'], color='coral', linewidth=1.5)
axes[1].set_title('Average Price Over Time')
axes[1].set_ylabel('Price (USD)')

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:
"""### 3.4 Correlation Analysis"""

In [ ]:
# correlation heatmap for numeric columns
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# what correlates most with deliveries?
print("Correlations with Estimated_Deliveries:")
print(corr_matrix['Estimated_Deliveries'].sort_values(ascending=False))

In [ ]:
"""---
## Part 4 — Feature Engineering
"""

In [ ]:
# 1. create new time-based features
df['Quarter'] = df['Month'].apply(lambda m: (m - 1) // 3 + 1)
df['Is_Q4'] = (df['Quarter'] == 4).astype(int)  # holiday quarter
df['Half'] = (df['Month'] > 6).astype(int)  # first or second half of year

In [ ]:
# 2. price per km range - kinda like value for money
df['Price_per_km'] = df['Avg_Price_USD'] / df['Range_km']

In [ ]:
# 3. production efficiency ratio
df['Delivery_Rate'] = df['Estimated_Deliveries'] / df['Production_Units']

In [ ]:
# 4. CO2 efficiency
df['CO2_per_delivery'] = df['CO2_Saved_tons'] / df['Estimated_Deliveries']

In [ ]:
# 5. stations per delivery (infrastructure ratio)
df['Stations_per_delivery'] = df['Charging_Stations'] / df['Estimated_Deliveries']

In [ ]:
# 6. Encode categorical variables
le_region = LabelEncoder()
le_model = LabelEncoder()
le_source = LabelEncoder()

In [ ]:
df['Region_encoded'] = le_region.fit_transform(df['Region'])
df['Model_encoded'] = le_model.fit_transform(df['Model'])
df['Source_encoded'] = le_source.fit_transform(df['Source_Type'])

In [ ]:
print("New features created:")
print(df[['Quarter', 'Is_Q4', 'Price_per_km', 'Delivery_Rate', 'CO2_per_delivery']].head(10))
print(f"\nUpdated shape: {df.shape}")
print(f"All columns: {list(df.columns)}")

In [ ]:
"""### 4.1 Feature Importance (Quick look with Random Forest)"""

In [ ]:
# quick feature importance check
feature_cols = ['Year', 'Month', 'Quarter', 'Is_Q4', 'Half',
                'Production_Units', 'Avg_Price_USD', 'Battery_Capacity_kWh',
                'Range_km', 'Charging_Stations', 'Region_encoded',
                'Model_encoded', 'Source_encoded',
                'Price_per_km', 'Delivery_Rate']

In [ ]:
X_temp = df[feature_cols]
y_temp = df['Estimated_Deliveries']

In [ ]:
rf_temp = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_temp.fit(X_temp, y_temp)

In [ ]:
importances = pd.Series(rf_temp.feature_importances_, index=feature_cols).sort_values(ascending=True)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
importances.plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Feature Importances (Random Forest)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
print("\nTop 5 features:")
print(importances.tail(5))

In [ ]:
"""---
## Part 5 — Regression Modeling

### 5.1 Prepare Data
"""

In [ ]:
# target: predict Estimated_Deliveries
# removing features that would leak information (like CO2 saved which is directly derived from deliveries)
features = ['Year', 'Month', 'Quarter', 'Is_Q4', 'Half',
            'Production_Units', 'Avg_Price_USD', 'Battery_Capacity_kWh',
            'Range_km', 'Charging_Stations', 'Region_encoded',
            'Model_encoded', 'Source_encoded', 'Price_per_km']

In [ ]:
X = df[features]
y = df['Estimated_Deliveries']

In [ ]:
# train test split - 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
"""### 5.2 Train Multiple Models"""

In [ ]:
# helper function to evaluate models
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    mae = mean_absolute_error(y_te, y_pred)
    mse = mean_squared_error(y_te, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_te, y_pred)

    print(f"\n--- {name} ---")
    print(f"MAE:  {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2:   {r2:.4f}")

    return {'name': name, 'mae': mae, 'rmse': rmse, 'r2': r2, 'predictions': y_pred}

In [ ]:
# 1. Linear Regression
results = []
res = evaluate_model("Linear Regression", LinearRegression(),
                     X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

In [ ]:
# 2. Ridge Regression
res = evaluate_model("Ridge Regression", Ridge(alpha=1.0),
                     X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

In [ ]:
# 3. Lasso Regression
res = evaluate_model("Lasso Regression", Lasso(alpha=1.0),
                     X_train_scaled, X_test_scaled, y_train, y_test)
results.append(res)

In [ ]:
# 4. Random Forest
res = evaluate_model("Random Forest", RandomForestRegressor(n_estimators=100, random_state=42),
                     X_train, X_test, y_train, y_test)  # RF doesnt need scaling
results.append(res)

In [ ]:
# 5. Gradient Boosting
res = evaluate_model("Gradient Boosting", GradientBoostingRegressor(n_estimators=100, random_state=42),
                     X_train, X_test, y_train, y_test)
results.append(res)

In [ ]:
"""### 5.3 Model Comparison"""

In [ ]:
results_df = pd.DataFrame(results)[['name', 'mae', 'rmse', 'r2']]
print("\n=== Model Comparison ===")
print(results_df.to_string(index=False))

In [ ]:
# plot comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

In [ ]:
names = results_df['name']
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']

In [ ]:
axes[0].barh(names, results_df['mae'], color=colors)
axes[0].set_title('MAE (lower is better)')
axes[0].set_xlabel('MAE')

In [ ]:
axes[1].barh(names, results_df['rmse'], color=colors)
axes[1].set_title('RMSE (lower is better)')
axes[1].set_xlabel('RMSE')

In [ ]:
axes[2].barh(names, results_df['r2'], color=colors)
axes[2].set_title('R² Score (higher is better)')
axes[2].set_xlabel('R²')

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:
# best model
best = results_df.loc[results_df['r2'].idxmax()]
print(f"\nBest model: {best['name']} with R² = {best['r2']:.4f}")

In [ ]:
"""### 5.4 Actual vs Predicted Plot (Best Model)"""

In [ ]:
# using the best tree-based model for plotting
# retrain the best one
best_model_name = best['name']
# find the predictions from results
best_preds = [r['predictions'] for r in results if r['name'] == best_model_name][0]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, best_preds, alpha=0.4, color='steelblue', edgecolors='k', linewidth=0.5)
# perfect prediction line
min_val = min(y_test.min(), best_preds.min())
max_val = max(y_test.max(), best_preds.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
ax.set_xlabel('Actual Deliveries')
ax.set_ylabel('Predicted Deliveries')
ax.set_title(f'Actual vs Predicted — {best_model_name}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# residuals plot
residuals = y_test.values - best_preds
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

In [ ]:
axes[0].scatter(best_preds, residuals, alpha=0.4, color='coral')
axes[0].axhline(y=0, color='black', linestyle='--')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted')

In [ ]:
axes[1].hist(residuals, bins=30, color='mediumpurple', edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual')

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:
"""---
## Part 6 — Hyperparameter Tuning

### 6.1 Grid Search on Gradient Boosting
"""

In [ ]:
# gonna tune Gradient Boosting since it usually performs well
# keeping the grid small so it doesnt take forever
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'min_samples_split': [2, 5]
}

In [ ]:
gb_model = GradientBoostingRegressor(random_state=42)

In [ ]:
# using 3-fold CV to save time
grid_search = GridSearchCV(gb_model, param_grid, cv=3, scoring='r2',
                           n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

In [ ]:
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV R² score: {grid_search.best_score_:.4f}")

In [ ]:
# evaluate on test set
best_gb = grid_search.best_estimator_
y_pred_tuned = best_gb.predict(X_test)

In [ ]:
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
r2_tuned = r2_score(y_test, y_pred_tuned)

In [ ]:
print(f"\n--- Tuned Gradient Boosting (Test Set) ---")
print(f"MAE:  {mae_tuned:.2f}")
print(f"RMSE: {rmse_tuned:.2f}")
print(f"R²:   {r2_tuned:.4f}")

In [ ]:
"""### 6.2 Cross-Validation Scores"""

In [ ]:
# cross validation on the tuned model
cv_scores = cross_val_score(best_gb, X_train, y_train, cv=5, scoring='r2')
print(f"\n5-Fold CV R² scores: {cv_scores}")
print(f"Mean CV R²: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

In [ ]:
"""### 6.3 Tuned Model — Actual vs Predicted"""

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, y_pred_tuned, alpha=0.4, color='mediumseagreen', edgecolors='k', linewidth=0.5)
min_val = min(y_test.min(), y_pred_tuned.min())
max_val = max(y_test.max(), y_pred_tuned.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
ax.set_xlabel('Actual')
ax.set_ylabel('Predicted')
ax.set_title(f'Tuned Gradient Boosting — R² = {r2_tuned:.4f}')
plt.tight_layout()
plt.show()

In [ ]:
"""---
## Part 7 — Time Series Forecasting

### 7.1 Prepare Time Series Data
"""

In [ ]:
# aggregate monthly total deliveries
ts_data = df.groupby('Date')['Estimated_Deliveries'].sum().reset_index()
ts_data = ts_data.sort_values('Date')
ts_data.set_index('Date', inplace=True)

In [ ]:
print(f"Time series shape: {ts_data.shape}")
print(f"Date range: {ts_data.index.min()} to {ts_data.index.max()}")
print(ts_data.head(10))

In [ ]:
# plot the time series
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ts_data.index, ts_data['Estimated_Deliveries'], color='steelblue', linewidth=1.2)
ax.set_title('Monthly Total Deliveries — Time Series')
ax.set_ylabel('Total Deliveries')
ax.set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
"""### 7.2 Stationarity Check"""

In [ ]:
from statsmodels.tsa.stattools import adfuller

In [ ]:
# ADF test on original series
result = adfuller(ts_data['Estimated_Deliveries'])
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")
print(f"Stationary? {result[1] < 0.05}")

In [ ]:
# if not stationary, difference it
if result[1] >= 0.05:
    ts_diff = ts_data['Estimated_Deliveries'].diff().dropna()
    result_diff = adfuller(ts_diff)
    print(f"\nAfter differencing:")
    print(f"ADF Statistic: {result_diff[0]:.4f}")
    print(f"p-value: {result_diff[1]:.4f}")
    print(f"Stationary now? {result_diff[1] < 0.05}")

In [ ]:
"""### 7.3 Moving Average & Exponential Smoothing"""

In [ ]:
# calculate rolling mean and std
window = 6  # 6 month window
ts_data['Rolling_Mean'] = ts_data['Estimated_Deliveries'].rolling(window=window).mean()
ts_data['Rolling_Std'] = ts_data['Estimated_Deliveries'].rolling(window=window).std()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ts_data.index, ts_data['Estimated_Deliveries'], label='Original', alpha=0.6)
ax.plot(ts_data.index, ts_data['Rolling_Mean'], label=f'{window}-Month Rolling Mean', color='red', linewidth=2)
ax.fill_between(ts_data.index,
                ts_data['Rolling_Mean'] - ts_data['Rolling_Std'],
                ts_data['Rolling_Mean'] + ts_data['Rolling_Std'],
                alpha=0.15, color='red', label='Rolling Std Band')
ax.set_title('Moving Average Analysis')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Simple Exponential Smoothing
from statsmodels.tsa.holtwinters import ExponentialSmoothing

In [ ]:
# split into train/test for time series (last 12 months for test)
train_ts = ts_data['Estimated_Deliveries'][:-12]
test_ts = ts_data['Estimated_Deliveries'][-12:]

In [ ]:
# fit exponential smoothing - using additive trend
model_es = ExponentialSmoothing(train_ts, trend='add', seasonal=None,
                                 initialization_method='estimated')
fit_es = model_es.fit()
pred_es = fit_es.forecast(12)

In [ ]:
print(f"Exponential Smoothing RMSE: {np.sqrt(mean_squared_error(test_ts, pred_es)):.2f}")

In [ ]:
"""### 7.4 ARIMA / SARIMAX Forecasting"""

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
# fit SARIMAX model
# using (1,1,1) order and (1,1,1,12) seasonal order
# these are reasonable starting params
try:
    model_sarima = SARIMAX(train_ts, order=(1, 1, 1),
                            seasonal_order=(1, 1, 1, 12),
                            enforce_stationarity=False,
                            enforce_invertibility=False)
    fit_sarima = model_sarima.fit(disp=False)
    print(fit_sarima.summary().tables[1])

    pred_sarima = fit_sarima.forecast(steps=12)
    rmse_sarima = np.sqrt(mean_squared_error(test_ts, pred_sarima))
    print(f"\nSARIMAX RMSE: {rmse_sarima:.2f}")
except Exception as e:
    print(f"SARIMAX failed (might need more data or different params): {e}")
    # fallback - just use a simpler ARIMA
    from statsmodels.tsa.arima.model import ARIMA
    model_arima = ARIMA(train_ts, order=(2, 1, 2))
    fit_sarima = model_arima.fit()
    pred_sarima = fit_sarima.forecast(steps=12)
    rmse_sarima = np.sqrt(mean_squared_error(test_ts, pred_sarima))
    print(f"ARIMA(2,1,2) RMSE: {rmse_sarima:.2f}")

In [ ]:
"""### 7.5 Forecast Visualization"""

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

In [ ]:
# plot training data
ax.plot(train_ts.index, train_ts.values, label='Training Data', color='steelblue')
# actual test data
ax.plot(test_ts.index, test_ts.values, label='Actual (Test)', color='black', linewidth=2)
# exp smoothing forecast
ax.plot(test_ts.index, pred_es.values, label='Exp. Smoothing', color='coral',
        linestyle='--', linewidth=2)
# sarima forecast
ax.plot(test_ts.index, pred_sarima.values, label='SARIMAX', color='mediumseagreen',
        linestyle='--', linewidth=2)

In [ ]:
ax.set_title('Time Series Forecasting — Last 12 Months')
ax.set_ylabel('Total Deliveries')
ax.set_xlabel('Date')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# compare forecasting methods
print("\n=== Forecasting Comparison ===")
rmse_es = np.sqrt(mean_squared_error(test_ts, pred_es))
print(f"Exponential Smoothing RMSE: {rmse_es:.2f}")
print(f"SARIMAX RMSE: {rmse_sarima:.2f}")

In [ ]:
"""---
## Part 8 — Summary & Conclusions

### Key Findings:

1. **Data Overview**: The dataset contains Tesla delivery records across multiple regions (North America, Europe, Asia, Middle East) for 5 models (Model S, 3, X, Y, Cybertruck) from 2015-2025.

2. **EDA Insights**:
   - Production_Units has the strongest correlation with deliveries (expected since they're closely related)
   - Avg price varies significantly by model and region
   - There's a general upward trend in deliveries over the years

3. **Feature Engineering**:
   - Created time-based features (Quarter, Half-year)
   - Derived features like Price_per_km and Delivery_Rate added predictive value
   - Label encoding for categorical features (Region, Model, Source_Type)

4. **Regression Results**:
   - Tree-based models (Random Forest, Gradient Boosting) outperformed linear models
   - Production_Units was the most important predictor
   - Hyperparameter tuning improved Gradient Boosting performance

5. **Time Series Forecasting**:
   - The series required differencing to achieve stationarity
   - SARIMAX with seasonal component captured patterns better than simple exponential smoothing
   - Both methods provide reasonable 12-month forecasts

### What I'd do differently next time:
- Try XGBoost or LightGBM for potentially better regression results
- Use auto_arima from pmdarima to automatically find best ARIMA parameters
- Add external features like economic indicators or EV market data
- Try prophet for time series as an alternative
"""

In [ ]:
print("\n" + "="*50)
print("Week 2 Assignment Complete!")
print("="*50)